In [1]:
from langchain_community.document_loaders import DirectoryLoader , PyMuPDFLoader

loader=DirectoryLoader(
    "PDFS",
    glob="./*.pdf",
    loader_cls=PyMuPDFLoader
)

raw_documents=loader.load()

C:\Users\Admin\AppData\Local\Temp\ipykernel_25236\2862539797.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader , PyMuPDFLoader


In [2]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings


# Embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Semantic Chunker
text_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=75.0
)


# Create chunks
chunks = text_splitter.split_documents(raw_documents)

print("Number of chunks:", len(chunks))


# Display chunks
for i, chunk in enumerate(chunks):
    print(f"\n---- Chunk {i + 1} ----")
    print(chunk.page_content.strip())

C:\Users\Admin\AppData\Local\Temp\ipykernel_25236\878114669.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 16

---- Chunk 1 ----
Artificial Intelligence in Healthcare
Informative Overview
Introduction
Artificial intelligence is increasingly becoming a foundational technology in modern healthcare. Rather than replacing doctors, nurses, researchers, and technicians, AI systems are primarily
developed to assist with tasks involving large amounts of data, pattern recognition, prediction,
automation, and decision support. Healthcare organizations generate enormous volumes of
information through electronic health records, medical images, laboratory tests, wearable devices,
genomic sequencing, insurance claims, and clinical research. The scale and complexity of this
information make computational assistance increasingly valuable. Medical Imaging and Diagnosis
One of the most visible applications of AI is medical imaging. Machine-learning and deep-learning
models can analyze X-rays, CT scans, MRI images, retinal photographs, pathology slides, and
other clinical imagery. Modern vis

In [3]:
from langchain_chroma import Chroma

db=Chroma(
    collection_name="all_data",
    embedding_function=embeddings,
    persist_directory="./ch_db_trial"
    
)
db.add_documents(documents=chunks)


['ae32766d-43f3-4bd6-849b-308f3e7c976d',
 'bd7f9b00-796c-4ced-a99d-f20f9aacd9c9',
 'fb5a4949-f5ef-4321-a241-5c9f7397bac9',
 '8f6be502-23e1-4818-9d12-c6e1642b35f8',
 '5f0063ff-e25c-4039-8679-cd34a8e51fc4',
 'e49d6eeb-ce3e-4bb0-9d2d-9c18841b0d97',
 'd93ba87e-44d2-443e-8390-3e5689e55f78',
 '1e6845d3-e0be-44bc-82cb-5e9db3b052b7',
 '5e34e2b9-fe97-400e-8e1e-3dbc526edf99',
 '3107c99f-fc7a-41ad-98cf-4625d9c5f7c3',
 '46d237a7-6d6a-4f75-889d-b661e97b00e7',
 'a5288f79-5602-4ce6-b05b-68a3fca96c1c',
 'b212e2cd-cfe2-4a90-88a9-4d37a767a9c7',
 '40d3aa0b-e2da-4d3a-afe3-d612f3c90288',
 '795cfc47-77bc-40b1-aef7-0c3cb56f2594',
 'd878ece4-7d64-44da-a22c-c9409ffeef71']

In [ ]:
'''
dense_retriever=db.as_retriever(
    search_kwargs={"k":10}
)
'''

In [19]:
mmr_retriver=db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10,
        "lambda_mult":0.5
    }
)

In [25]:
query = "How is AI used in video games?"

mmr_results = mmr_retriver.invoke(query)


print("\n===== MMR =====")

for i, doc in enumerate(mmr_results):
    print(f"\n--- Result {i + 1} ---")
    print(doc.page_content)


===== MMR =====

--- Result 1 ---
The Evolution of Video Games and Artificial
Intelligence
Informative Overview
Introduction
Video games have evolved from simple electronic experiments into a major form of interactive
media. Early games were constrained by limited memory, processing power, storage, graphics
hardware, and input devices. Modern games can simulate large worlds, render complex scenes in
real time, connect millions of players, and support sophisticated physical and behavioral systems. Artificial intelligence has played an important role in this evolution, particularly in controlling
non-player characters, generating content, adapting difficulty, and supporting development
workflows. Traditional Game AI
Many successful game-AI systems do not rely on machine learning. Developers commonly use
finite-state machines, behavior trees, utility systems, navigation algorithms, scripted triggers, and
rule-based logic.

--- Result 2 ---
Machine Learning and Reinforcement Learning
Mach

In [5]:
from langchain_community.retrievers import BM25Retriever

sparse_retriever=BM25Retriever.from_documents(chunks)
sparse_retriever.k=3

In [6]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3]
)




In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\Admin\Desktop\Python_practice\ALL_STACK_ENV\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [18]:
query = "How is reinforcement learning used in video games?"

candidates = hybrid_retriever.invoke(query)

pairs=[
    [query,doc.page_content]
    for doc in candidates
]

scores=reranker.predict(pairs)

ranked_res=sorted(
    zip(scores,candidates),
    key=lambda x:x[0],
    reverse=True
)

for i , (score , doc) in enumerate(ranked_res):
    print(f"\n--- Result {i + 1} | Score: {score:.4f} ---")
    print(doc.page_content)


--- Result 1 | Score: 5.6431 ---
Machine Learning and Reinforcement Learning
Machine learning provides another way to create behavior. Instead of manually specifying every
rule, developers can train models using examples or interaction. Reinforcement learning is
especially interesting for games because an agent can learn through repeated attempts to
maximize a reward. Game environments are valuable research platforms because they provide
measurable objectives and controllable simulations.

--- Result 2 | Score: -1.5332 ---
The Evolution of Video Games and Artificial
Intelligence
Informative Overview
Introduction
Video games have evolved from simple electronic experiments into a major form of interactive
media. Early games were constrained by limited memory, processing power, storage, graphics
hardware, and input devices. Modern games can simulate large worlds, render complex scenes in
real time, connect millions of players, and support sophisticated physical and behavioral systems. Ar

In [9]:
query="what are the application of AI in Games"

result=db.similarity_search(query,k=3)

for i, result in enumerate(result):
    print(f"\n--- Result {i+1} ---")
    print(result.page_content)


--- Result 1 ---
The Evolution of Video Games and Artificial
Intelligence
Informative Overview
Introduction
Video games have evolved from simple electronic experiments into a major form of interactive
media. Early games were constrained by limited memory, processing power, storage, graphics
hardware, and input devices. Modern games can simulate large worlds, render complex scenes in
real time, connect millions of players, and support sophisticated physical and behavioral systems. Artificial intelligence has played an important role in this evolution, particularly in controlling
non-player characters, generating content, adapting difficulty, and supporting development
workflows. Traditional Game AI
Many successful game-AI systems do not rely on machine learning. Developers commonly use
finite-state machines, behavior trees, utility systems, navigation algorithms, scripted triggers, and
rule-based logic.

--- Result 2 ---
The Evolution of Video Games and Artificial
Intelligence
Informat

In [10]:
queries = [
    "How is AI used in healthcare?",
    "What are the applications of AI in medical imaging?",
    "How is reinforcement learning used in video games?",
    "What are traditional game AI techniques?",
    "What are the challenges of AI in healthcare?"
]

for query in queries:
    print(f"\n\nQUERY: {query}")

    results = db.similarity_search(query, k=5)

    for i, result in enumerate(results):
        print(f"\n--- Result {i+1} ---")
        print(result.page_content[:500])



QUERY: How is AI used in healthcare?

--- Result 1 ---
different from their training data. Regulation, clinical validation, privacy, security, auditability, data
governance, and accountability are essential. The most promising future is likely to involve
collaboration in which AI handles specialized computational tasks while trained professionals retain
responsibility for interpretation and patient care. Conclusion
This topic demonstrates that major technological and social transformations depend on more than
technical capability alone.

--- Result 2 ---
different from their training data. Regulation, clinical validation, privacy, security, auditability, data
governance, and accountability are essential. The most promising future is likely to involve
collaboration in which AI handles specialized computational tasks while trained professionals retain
responsibility for interpretation and patient care. Conclusion
This topic demonstrates that major technological and social transformatio